In [2]:
import os
import re
import unicodedata
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd
import numpy as np

BASE_DIR = ".."

TRAIN_DIR = os.path.join(BASE_DIR, "data", "train")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOURCE1 = os.path.join(TRAIN_DIR, "train_source1.tsv")
SOURCE2 = os.path.join(TRAIN_DIR, "train_source2.tsv")
SOURCE3 = os.path.join(TRAIN_DIR, "train_source3.tsv")
GROUND_TRUTH = os.path.join(TRAIN_DIR, "train_ground_truth.tsv")

VALIDATION_IDS_FILE = os.path.join(
    OUTPUT_DIR,
    "validation_s1_ids.txt"
)

CHUNK_SIZE = 100_000

PAIR_THRESHOLDS = [2, 5, 10, 20, 50, 100]
MAX_PAIR_FREQUENCY = max(PAIR_THRESHOLDS)

print("Thresholds:", PAIR_THRESHOLDS)

Thresholds: [2, 5, 10, 20, 50, 100]


In [3]:
def normalize_text(value):
    value = "" if value is None else str(value)

    value = unicodedata.normalize("NFKC", value)
    value = value.casefold()

    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
        flags=re.UNICODE
    )

    return value.strip()


def get_tokens(value):
    value = normalize_text(value)

    if not value:
        return set()

    return {
        token
        for token in value.split()
        if len(token) >= 2
    }

In [4]:
with open(VALIDATION_IDS_FILE, "r", encoding="utf-8") as f:
    validation_s1_ids = {
        line.strip()
        for line in f
        if line.strip()
    }

validation_s1_parts = []

for chunk in pd.read_csv(
    SOURCE1,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):
    selected = chunk[
        chunk["entity_id"].isin(validation_s1_ids)
    ]

    if len(selected) > 0:
        validation_s1_parts.append(selected)

validation_s1 = pd.concat(
    validation_s1_parts,
    ignore_index=True
)

validation_s1 = validation_s1.rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

validation_s1 = validation_s1.reset_index(drop=True)

print("Validation S1 rows:", len(validation_s1))

Validation S1 rows: 220682


In [5]:
s1_id_to_index = {
    s1_id: i
    for i, s1_id in enumerate(
        validation_s1["s1_id"]
    )
}

print(
    "S1 index entries:",
    len(s1_id_to_index)
)

S1 index entries: 220682


In [6]:
token_to_id = {}
next_token_id = 1

name_pair_frequency = Counter()
address_pair_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    # -----------------------
    # NAME
    # -----------------------
    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    name_token_ids = []

    for token in name_tokens:

        if token not in token_to_id:
            token_to_id[token] = next_token_id
            next_token_id += 1

        name_token_ids.append(
            token_to_id[token]
        )

    if len(name_token_ids) >= 2:
        name_pair_frequency.update(
            combinations(name_token_ids, 2)
        )

    # -----------------------
    # ADDRESS
    # -----------------------
    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    address_token_ids = []

    for token in address_tokens:

        if token not in token_to_id:
            token_to_id[token] = next_token_id
            next_token_id += 1

        address_token_ids.append(
            token_to_id[token]
        )

    if len(address_token_ids) >= 2:
        address_pair_frequency.update(
            combinations(address_token_ids, 2)
        )

print("Unique tokens:", len(token_to_id))
print("Unique name pairs:", len(name_pair_frequency))
print("Unique address pairs:", len(address_pair_frequency))

Unique tokens: 146321
Unique name pairs: 385258
Unique address pairs: 2472984


In [7]:
def frequency_summary(counter):
    values = np.array(
        list(counter.values()),
        dtype=np.int64
    )

    return pd.Series(
        values
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )


print("NAME PAIR FREQUENCY")
display(
    frequency_summary(name_pair_frequency)
)

print("\nADDRESS PAIR FREQUENCY")
display(
    frequency_summary(address_pair_frequency)
)

NAME PAIR FREQUENCY


count    385258.000000
mean          2.649959
std          74.457034
min           1.000000
50%           1.000000
75%           1.000000
90%           3.000000
95%           6.000000
99%          18.000000
max       43134.000000
dtype: float64


ADDRESS PAIR FREQUENCY


count    2.472984e+06
mean     2.621238e+00
std      2.961950e+01
min      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      3.000000e+00
95%      5.000000e+00
99%      2.300000e+01
max      8.872000e+03
dtype: float64

In [8]:
name_pair_to_s1 = defaultdict(list)
address_pair_to_s1 = defaultdict(list)

for s1_idx, row in enumerate(
    validation_s1.itertuples(index=False)
):

    # -----------------------
    # NAME PAIRS
    # -----------------------
    name_token_ids = []

    for token in sorted(
        get_tokens(row.s1_name)
    ):
        name_token_ids.append(
            token_to_id[token]
        )

    if len(name_token_ids) >= 2:

        for pair in combinations(
            name_token_ids,
            2
        ):

            if (
                name_pair_frequency.get(pair, 0)
                <= MAX_PAIR_FREQUENCY
            ):
                name_pair_to_s1[pair].append(
                    s1_idx
                )

    # -----------------------
    # ADDRESS PAIRS
    # -----------------------
    address_token_ids = []

    for token in sorted(
        get_tokens(row.s1_address)
    ):
        address_token_ids.append(
            token_to_id[token]
        )

    if len(address_token_ids) >= 2:

        for pair in combinations(
            address_token_ids,
            2
        ):

            if (
                address_pair_frequency.get(pair, 0)
                <= MAX_PAIR_FREQUENCY
            ):
                address_pair_to_s1[pair].append(
                    s1_idx
                )

print(
    "Indexed name pairs:",
    len(name_pair_to_s1)
)

print(
    "Indexed address pairs:",
    len(address_pair_to_s1)
)

Indexed name pairs: 384357
Indexed address pairs: 2468486


In [9]:
validation_ground_truth = {}

for chunk in pd.read_csv(
    GROUND_TRUTH,
    sep="\t",
    dtype=str,
    chunksize=CHUNK_SIZE,
    keep_default_na=False
):

    selected = chunk[
        chunk["source1_entity_id"].isin(
            validation_s1_ids
        )
    ]

    for row in selected.itertuples(index=False):

        value = row.matched_entity_ids.strip()

        if value:
            matches = {
                x.strip()
                for x in value.split(",")
                if x.strip()
            }
        else:
            matches = set()

        validation_ground_truth[
            row.source1_entity_id
        ] = matches


true_target_to_s1_idx = {}

for s1_id, target_ids in validation_ground_truth.items():

    s1_idx = s1_id_to_index[s1_id]

    for target_id in target_ids:

        true_target_to_s1_idx[target_id] = s1_idx


total_true_matches = len(
    true_target_to_s1_idx
)

print(
    "Validation true matches:",
    total_true_matches
)

Validation true matches: 764045


In [10]:
def evaluate_pair_blocking_source(
    source_path,
    source_name
):
    candidate_pairs = {
        threshold: 0
        for threshold in PAIR_THRESHOLDS
    }

    candidate_counts_per_s1 = {
        threshold: np.zeros(
            len(validation_s1),
            dtype=np.int64
        )
        for threshold in PAIR_THRESHOLDS
    }

    true_hits = {
        threshold: set()
        for threshold in PAIR_THRESHOLDS
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            target_id = row.entity_id

            # -----------------------
            # Convert target tokens to IDs
            # -----------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            # Build only the pairs that actually
            # exist in the validation S1 vocabulary.
            name_pairs = set(
                combinations(
                    sorted(name_token_ids),
                    2
                )
            )

            address_pairs = set(
                combinations(
                    sorted(address_token_ids),
                    2
                )
            )

            for threshold in PAIR_THRESHOLDS:

                candidate_s1 = set()

                # -----------------------
                # Name pair candidates
                # -----------------------
                for pair in name_pairs:

                    frequency = (
                        name_pair_frequency.get(
                            pair,
                            0
                        )
                    )

                    if (
                        0 < frequency <= threshold
                    ):
                        candidate_s1.update(
                            name_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                # -----------------------
                # Address pair candidates
                # -----------------------
                for pair in address_pairs:

                    frequency = (
                        address_pair_frequency.get(
                            pair,
                            0
                        )
                    )

                    if (
                        0 < frequency <= threshold
                    ):
                        candidate_s1.update(
                            address_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                count = len(candidate_s1)

                candidate_pairs[
                    threshold
                ] += count

                # Candidate load for each S1.
                for s1_idx in candidate_s1:
                    candidate_counts_per_s1[
                        threshold
                    ][s1_idx] += 1

                # True match recall
                if target_id in true_target_to_s1_idx:

                    true_s1_idx = (
                        true_target_to_s1_idx[
                            target_id
                        ]
                    )

                    if true_s1_idx in candidate_s1:

                        true_hits[
                            threshold
                        ].add(target_id)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    return {
        "candidate_pairs": candidate_pairs,
        "candidate_counts_per_s1": candidate_counts_per_s1,
        "true_hits": true_hits
    }

In [11]:
print("Starting S2...")

pair_results_s2 = evaluate_pair_blocking_source(
    SOURCE2,
    "S2"
)

print("S2 complete.")

Starting S2...
S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S2 complete.


In [12]:
print("Starting S3...")

pair_results_s3 = evaluate_pair_blocking_source(
    SOURCE3,
    "S3"
)

print("S3 complete.")

Starting S3...
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed
S3 complete.


In [13]:
pair_blocking_results = []

for threshold in PAIR_THRESHOLDS:

    true_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    total_candidates = (
        pair_results_s2["candidate_pairs"][threshold]
        +
        pair_results_s3["candidate_pairs"][threshold]
    )

    candidates_per_s1 = (
        pair_results_s2[
            "candidate_counts_per_s1"
        ][threshold]
        +
        pair_results_s3[
            "candidate_counts_per_s1"
        ][threshold]
    )

    pair_blocking_results.append({

        "pair_frequency_threshold": threshold,

        "candidate_pairs": total_candidates,

        "true_matches_captured": len(
            true_hits
        ),

        "total_true_matches": total_true_matches,

        "candidate_recall": (
            len(true_hits)
            / total_true_matches
        ),

        "avg_candidates_per_s1": (
            total_candidates
            / len(validation_s1)
        ),

        "median_candidates_per_s1": (
            np.median(candidates_per_s1)
        ),

        "p95_candidates_per_s1": (
            np.percentile(
                candidates_per_s1,
                95
            )
        ),

        "max_candidates_per_s1": (
            np.max(candidates_per_s1)
        )
    })


pair_blocking_results_df = pd.DataFrame(
    pair_blocking_results
)

pair_blocking_results_df

,pair_frequency_threshold,candidate_pairs,true_matches_captured,total_true_matches,candidate_recall,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1
0,2,18183570,627313,764045,0.821042,82.397160,35.0,256.00,96945
1,5,45313947,658646,764045,0.862051,205.335945,94.0,652.00,194200
2,10,88172492,676178,764045,0.884998,399.545464,205.0,1292.00,195890
3,20,169905281,688143,764045,0.900658,769.910011,405.0,2488.00,197397
4,50,366472574,697760,764045,0.913245,1660.636454,884.0,5682.95,202231
5,100,668259328,704326,764045,0.921838,3028.155119,1639.0,10586.00,203352


In [14]:
pair_blocking_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "blocking_experiment_02_pair_results.csv"
    ),
    index=False
)

print("Saved pair-blocking results.")

Saved pair-blocking results.


In [16]:
import pickle
import os

CAPTURED_PATH = os.path.join(
    OUTPUT_DIR,
    "blocking_threshold_10_captured.pkl"
)

with open(CAPTURED_PATH, "rb") as f:
    current_captured = pickle.load(f)

print(
    "Current blocker captured:",
    len(current_captured)
)

pair_results_comparison = []

for threshold in PAIR_THRESHOLDS:

    pair_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    overlap = (
        pair_hits
        &
        current_captured
    )

    new_from_pair = (
        pair_hits
        -
        current_captured
    )

    union_hits = (
        pair_hits
        |
        current_captured
    )

    pair_results_comparison.append({
        "pair_threshold": threshold,
        "pair_hits": len(pair_hits),
        "overlap_with_current": len(overlap),
        "new_matches_from_pair": len(new_from_pair),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

pair_comparison_df = pd.DataFrame(
    pair_results_comparison
)

pair_comparison_df

Current blocker captured: 572131


,pair_threshold,pair_hits,overlap_with_current,new_matches_from_pair,union_true_matches,union_recall
0,2,627313,502513,124800,696931,0.912160
1,5,658646,520147,138499,710630,0.930089
2,10,676178,529041,147137,719268,0.941395
3,20,688143,535111,153032,725163,0.949110
4,50,697760,540058,157702,729833,0.955223
5,100,704326,543363,160963,733094,0.959491


In [17]:
pair_results_comparison = []

for threshold in PAIR_THRESHOLDS:

    pair_hits = (
        pair_results_s2["true_hits"][threshold]
        |
        pair_results_s3["true_hits"][threshold]
    )

    overlap = (
        pair_hits
        &
        current_captured
    )

    new_from_pair = (
        pair_hits
        -
        current_captured
    )

    union_hits = (
        pair_hits
        |
        current_captured
    )

    pair_results_comparison.append({
        "pair_threshold": threshold,
        "pair_hits": len(pair_hits),
        "overlap_with_current": len(overlap),
        "new_matches_from_pair": len(new_from_pair),
        "union_true_matches": len(union_hits),
        "union_recall": (
            len(union_hits)
            / total_true_matches
        )
    })

pair_comparison_df = pd.DataFrame(
    pair_results_comparison
)

pair_comparison_df

,pair_threshold,pair_hits,overlap_with_current,new_matches_from_pair,union_true_matches,union_recall
0,2,627313,502513,124800,696931,0.912160
1,5,658646,520147,138499,710630,0.930089
2,10,676178,529041,147137,719268,0.941395
3,20,688143,535111,153032,725163,0.949110
4,50,697760,540058,157702,729833,0.955223
5,100,704326,543363,160963,733094,0.959491


In [18]:
total_true_matches = len(true_target_to_s1_idx)

print(
    "Total validation true matches:",
    total_true_matches
)

Total validation true matches: 764045


In [19]:
import pickle

pair_baseline_state = {
    "pair_true_hits": {
        threshold: (
            pair_results_s2["true_hits"][threshold]
            |
            pair_results_s3["true_hits"][threshold]
        )
        for threshold in PAIR_THRESHOLDS
    },
    "total_true_matches": total_true_matches
}

pair_state_path = os.path.join(
    OUTPUT_DIR,
    "blocking_experiment_02_pair_state.pkl"
)

with open(pair_state_path, "wb") as f:
    pickle.dump(
        pair_baseline_state,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", pair_state_path)

Saved: ../output/blocking_experiment_02_pair_state.pkl


In [20]:
UNION_THRESHOLDS = [2, 5, 10]

In [21]:
def evaluate_union_candidate_volume(
    source_path,
    source_name,
    thresholds=UNION_THRESHOLDS
):
    total_candidates = {
        threshold: 0
        for threshold in thresholds
    }

    candidate_counts_per_s1 = {
        threshold: np.zeros(
            len(validation_s1),
            dtype=np.int32
        )
        for threshold in thresholds
    }

    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            target_name_norm = normalize_text(
                row.business_name
            )

            target_address_norm = normalize_text(
                row.business_address
            )

            # -------------------------------------------------
            # Existing current blocker
            # exact normalized name
            # + name token <=10
            # + address token <=10
            # -------------------------------------------------
            current_candidates = set()

            if target_name_norm:
                current_candidates.update(
                    name_exact_map.get(
                        target_name_norm,
                        []
                    )
                )

            for token in get_tokens(row.business_name):

                freq = name_token_frequency.get(
                    token,
                    0
                )

                if 0 < freq <= 10:
                    current_candidates.update(
                        name_token_to_s1.get(
                            token,
                            []
                        )
                    )

            for token in get_tokens(row.business_address):

                freq = address_token_frequency.get(
                    token,
                    0
                )

                if 0 < freq <= 10:
                    current_candidates.update(
                        address_token_to_s1.get(
                            token,
                            []
                        )
                    )

            # -------------------------------------------------
            # Pair candidates
            # -------------------------------------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            name_pairs = set(
                combinations(
                    sorted(name_token_ids),
                    2
                )
            )

            address_pairs = set(
                combinations(
                    sorted(address_token_ids),
                    2
                )
            )

            for threshold in thresholds:

                union_candidates = set(
                    current_candidates
                )

                # Name pair candidates
                for pair in name_pairs:

                    freq = name_pair_frequency.get(
                        pair,
                        0
                    )

                    if 0 < freq <= threshold:
                        union_candidates.update(
                            name_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                # Address pair candidates
                for pair in address_pairs:

                    freq = address_pair_frequency.get(
                        pair,
                        0
                    )

                    if 0 < freq <= threshold:
                        union_candidates.update(
                            address_pair_to_s1.get(
                                pair,
                                []
                            )
                        )

                count = len(union_candidates)

                total_candidates[
                    threshold
                ] += count

                for s1_idx in union_candidates:
                    candidate_counts_per_s1[
                        threshold
                    ][s1_idx] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    return {
        "total_candidates": total_candidates,
        "candidate_counts_per_s1": candidate_counts_per_s1
    }

In [31]:
from collections import defaultdict

# ---------------------------------------------------------
# Rebuild ALL candidate indexes using integer S1 indices
# ---------------------------------------------------------

name_exact_map = defaultdict(list)

name_token_to_s1 = defaultdict(list)
address_token_to_s1 = defaultdict(list)

name_pair_to_s1 = defaultdict(list)
address_pair_to_s1 = defaultdict(list)


for s1_idx, row in enumerate(
    validation_s1.itertuples(index=False)
):

    # =====================================================
    # NAME
    # =====================================================

    name_norm = normalize_text(
        row.s1_name
    )

    if name_norm:
        name_exact_map[
            name_norm
        ].append(s1_idx)

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    for token in name_tokens:

        # Single-token blocking
        if token in name_token_frequency:
            name_token_to_s1[
                token
            ].append(s1_idx)

    # Name token pairs
    if len(name_tokens) >= 2:

        name_ids = [
            token_to_id[token]
            for token in name_tokens
            if token in token_to_id
        ]

        for pair in combinations(
            sorted(name_ids),
            2
        ):

            frequency = name_pair_frequency.get(
                pair,
                0
            )

            if (
                0 < frequency <= MAX_PAIR_FREQUENCY
            ):
                name_pair_to_s1[
                    pair
                ].append(s1_idx)


    # =====================================================
    # ADDRESS
    # =====================================================

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    for token in address_tokens:

        # Single-token blocking
        if token in address_token_frequency:
            address_token_to_s1[
                token
            ].append(s1_idx)

    # Address token pairs
    if len(address_tokens) >= 2:

        address_ids = [
            token_to_id[token]
            for token in address_tokens
            if token in token_to_id
        ]

        for pair in combinations(
            sorted(address_ids),
            2
        ):

            frequency = address_pair_frequency.get(
                pair,
                0
            )

            if (
                0 < frequency <= MAX_PAIR_FREQUENCY
            ):
                address_pair_to_s1[
                    pair
                ].append(s1_idx)


print("Exact-name keys:", len(name_exact_map))
print("Name-token keys:", len(name_token_to_s1))
print("Address-token keys:", len(address_token_to_s1))
print("Name-pair keys:", len(name_pair_to_s1))
print("Address-pair keys:", len(address_pair_to_s1))

Exact-name keys: 187550
Name-token keys: 48652
Address-token keys: 118619
Name-pair keys: 193741
Address-pair keys: 1004714


In [32]:
maps_to_check = {
    "name_exact_map": name_exact_map,
    "name_token_to_s1": name_token_to_s1,
    "address_token_to_s1": address_token_to_s1,
    "name_pair_to_s1": name_pair_to_s1,
    "address_pair_to_s1": address_pair_to_s1,
}

for map_name, mapping in maps_to_check.items():

    sample_values = []

    for values in mapping.values():
        if values:
            sample_values.extend(values[:3])

        if len(sample_values) >= 3:
            break

    print(
        f"{map_name:25} ->",
        [type(x).__name__ for x in sample_values],
        sample_values
    )

name_exact_map            -> ['int', 'int', 'int'] [0, 1, 2]
name_token_to_s1          -> ['int', 'int', 'int'] [0, 6, 414]
address_token_to_s1       -> ['int', 'int', 'int'] [0, 2838, 13546]
name_pair_to_s1           -> ['int', 'int', 'int'] [0, 1424, 7001]
address_pair_to_s1        -> ['int', 'int', 'int'] [0, 25043, 51217]


In [29]:
sample_key = next(iter(name_exact_map))

print(
    "Sample key:",
    sample_key
)

print(
    "Sample mapped values:",
    name_exact_map[sample_key][:5]
)

print(
    "Types:",
    [type(x).__name__ for x in name_exact_map[sample_key][:5]]
)

Sample key: consulting nyasa nursing private limited
Sample mapped values: [0]
Types: ['int']


In [33]:
union_s2 = evaluate_union_candidate_volume(
    SOURCE2,
    "S2"
)


S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed


In [34]:
union_s3 = evaluate_union_candidate_volume(
    SOURCE3,
    "S3"
)

S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [36]:
union_volume_results = []

for threshold in UNION_THRESHOLDS:

    total_candidates = (
        union_s2["total_candidates"][threshold]
        +
        union_s3["total_candidates"][threshold]
    )

    candidates_per_s1 = (
        union_s2["candidate_counts_per_s1"][threshold]
        +
        union_s3["candidate_counts_per_s1"][threshold]
    )

    union_volume_results.append({
        "threshold": threshold,
        "candidate_pairs": total_candidates,
        "avg_candidates_per_s1": (
            total_candidates / len(validation_s1)
        ),
        "median_candidates_per_s1": np.median(
            candidates_per_s1
        ),
        "p95_candidates_per_s1": np.percentile(
            candidates_per_s1, 95
        ),
        "max_candidates_per_s1": np.max(
            candidates_per_s1
        )
    })

union_volume_df = pd.DataFrame(
    union_volume_results
)

union_volume_df

,threshold,candidate_pairs,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1
0,2,59235008,268.417941,182.0,679.0,263003
1,5,82752186,374.983850,260.0,888.0,263743
2,10,122895709,556.890499,382.0,1395.0,264194


In [37]:
final_blocking_tradeoff = (
    union_volume_df
    .merge(
        pair_comparison_df[
            [
                "pair_threshold",
                "union_true_matches",
                "union_recall"
            ]
        ],
        left_on="threshold",
        right_on="pair_threshold",
        how="left"
    )
    .drop(columns=["pair_threshold"])
)

final_blocking_tradeoff

,threshold,candidate_pairs,avg_candidates_per_s1,median_candidates_per_s1,p95_candidates_per_s1,max_candidates_per_s1,union_true_matches,union_recall
0,2,59235008,268.417941,182.0,679.0,263003,696931,0.912160
1,5,82752186,374.983850,260.0,888.0,263743,710630,0.930089
2,10,122895709,556.890499,382.0,1395.0,264194,719268,0.941395


In [38]:
candidate_tail = []

# Use threshold 2
counts_s2 = union_s2["candidate_counts_per_s1"][2]
counts_s3 = union_s3["candidate_counts_per_s1"][2]

combined_counts = counts_s2 + counts_s3

for threshold in [1000, 5000, 10000, 50000, 100000]:
    candidate_tail.append({
        "candidate_threshold": threshold,
        "s1_entities_above_threshold": int(
            (combined_counts > threshold).sum()
        ),
        "percentage_of_validation_s1": (
            (combined_counts > threshold).mean() * 100
        )
    })

candidate_tail_df = pd.DataFrame(candidate_tail)

candidate_tail_df

,candidate_threshold,s1_entities_above_threshold,percentage_of_validation_s1
0,1000,2581,1.169556
1,5000,209,0.094706
2,10000,120,0.054377
3,50000,41,0.018579
4,100000,4,0.001813


In [39]:
pair_hits_threshold_2 = (
    pair_results_s2["true_hits"][2]
    |
    pair_results_s3["true_hits"][2]
)

current_blocker_hits_threshold_2 = (
    pair_hits_threshold_2
    |
    current_captured
)

all_true_targets = set(
    true_target_to_s1_idx.keys()
)

missed_after_threshold_2 = (
    all_true_targets
    -
    current_blocker_hits_threshold_2
)

print(
    "Total true matches:",
    len(all_true_targets)
)

print(
    "Captured:",
    len(current_blocker_hits_threshold_2)
)

print(
    "Still missed:",
    len(missed_after_threshold_2)
)

print(
    "Recall:",
    len(current_blocker_hits_threshold_2)
    / len(all_true_targets)
)

Total true matches: 764045
Captured: 696931
Still missed: 67114
Recall: 0.91215962410591


In [40]:
import random

rng = random.Random(42)

sample_size = min(
    1000,
    len(missed_after_threshold_2)
)

sampled_remaining_misses = set(
    rng.sample(
        list(missed_after_threshold_2),
        sample_size
    )
)

print(
    "Sampled remaining misses:",
    len(sampled_remaining_misses)
)

Sampled remaining misses: 1000


In [42]:
threshold_2_counts = (
    union_s2["candidate_counts_per_s1"][2]
    +
    union_s3["candidate_counts_per_s1"][2]
)

heavy_s1 = pd.DataFrame({
    "s1_index": np.arange(len(threshold_2_counts)),
    "candidate_count": threshold_2_counts
})

heavy_s1 = heavy_s1.sort_values(
    "candidate_count",
    ascending=False
)

heavy_s1 = heavy_s1[
    heavy_s1["candidate_count"] > 5000
]

heavy_s1 = heavy_s1.merge(
    validation_s1[
        [
            "s1_id",
            "s1_name",
            "s1_address",
            "s1_country"
        ]
    ].reset_index().rename(
        columns={"index": "s1_index"}
    ),
    on="s1_index",
    how="left"
)

heavy_s1.head(50)

,s1_index,candidate_count,s1_id,s1_name,s1_address,s1_country
0,93829,263003,S1-781739349,Vizag Opticals Private Limited,"Shop 3Ab Jariwala Building, Mumbai City, Mumba...",India
1,191119,262924,S1-816511031,Innovative Industries Pvt Ltd,27/28 Floor 6Th Plot 132/134 Aishabai Building...,India
2,126576,262764,S1-368134184,Aes Tradecom Pvt Ltd,"176/1, Second Floor, Hari Nagar Ashram, Null, ...",India
3,168405,262724,S1-37009159,Lucknow Estates Pvt Ltd,"Lig-49, Aishbaag Pill Col, Old Labour Colony N...",India
4,217428,97247,S1-652967049,Future Silver Tech,"Sri Skanda No. 123, 1St Floor, 1St Cross, Aa C...",India
5,164768,95515,S1-353569058,Astron Enn Pvt Ltd,"Survey No.321/Ak, Village Miroli, Taluka: Dask...",India
6,36259,95232,S1-301584609,Ss Software Private Limited,"Gj-02 Sdf-7 Seepz, Maharashtra, Mumbai, Andher...",India
7,112005,93943,S1-492263052,Chiplun Pharmaceutical LLP,"#330, Whitecity Layout Opp To Blossom Internat...",India
8,78565,75896,S1-497090852,UA Technotronics Pvt Ltd,"Door No Tg Block 29, Jains Alpine Meadows Temp...",India
9,59280,74604,S1-189147360,Ambassador & Co,"Plot No. 63 To 70, Tg/F, Mohit Nagar Kakrola, ...",India


In [43]:
def token_frequency_details(value, frequency_map):
    tokens = get_tokens(value)

    return sorted(
        [
            (token, frequency_map.get(token, 0))
            for token in tokens
        ],
        key=lambda x: x[1],
        reverse=True
    )


heavy_s1["name_token_frequencies"] = (
    heavy_s1["s1_name"]
    .map(
        lambda x: token_frequency_details(
            x,
            name_token_frequency
        )
    )
)

heavy_s1["address_token_frequencies"] = (
    heavy_s1["s1_address"]
    .map(
        lambda x: token_frequency_details(
            x,
            address_token_frequency
        )
    )
)

heavy_s1[
    [
        "s1_id",
        "candidate_count",
        "s1_name",
        "s1_address",
        "s1_country",
        "name_token_frequencies",
        "address_token_frequencies"
    ]
].head(30)

,s1_id,candidate_count,s1_name,s1_address,s1_country,name_token_frequencies,address_token_frequencies
0,S1-781739349,263003,Vizag Opticals Private Limited,"Shop 3Ab Jariwala Building, Mumbai City, Mumba...",India,"[(limited, 52308), (private, 43192), (vizag, 8...","[(road, 45852), (maharashtra, 19232), (city, 1..."
1,S1-816511031,262924,Innovative Industries Pvt Ltd,27/28 Floor 6Th Plot 132/134 Aishabai Building...,India,"[(ltd, 14974), (pvt, 12246), (industries, 1913...","[(street, 31080), (maharashtra, 19232), (floor..."
2,S1-368134184,262764,Aes Tradecom Pvt Ltd,"176/1, Second Floor, Hari Nagar Ashram, Null, ...",India,"[(ltd, 14974), (pvt, 12246), (tradecom, 33), (...","[(floor, 16306), (nagar, 15610), (delhi, 12180..."
3,S1-37009159,262724,Lucknow Estates Pvt Ltd,"Lig-49, Aishbaag Pill Col, Old Labour Colony N...",India,"[(ltd, 14974), (pvt, 12246), (estates, 127), (...","[(pradesh, 11329), (uttar, 7370), (colony, 508..."
4,S1-652967049,97247,Future Silver Tech,"Sri Skanda No. 123, 1St Floor, 1St Cross, Aa C...",India,"[(tech, 1372), (silver, 1222), (future, 449)]","[(road, 45852), (no, 36025), (floor, 16306), (..."
5,S1-353569058,95515,Astron Enn Pvt Ltd,"Survey No.321/Ak, Village Miroli, Taluka: Dask...",India,"[(ltd, 14974), (pvt, 12246), (astron, 5), (enn...","[(no, 36025), (gujarat, 5568), (in, 5143), (vi..."
6,S1-301584609,95232,Ss Software Private Limited,"Gj-02 Sdf-7 Seepz, Maharashtra, Mumbai, Andher...",India,"[(limited, 52308), (private, 43192), (software...","[(maharashtra, 19232), (mumbai, 8730), (andher..."
7,S1-492263052,93943,Chiplun Pharmaceutical LLP,"#330, Whitecity Layout Opp To Blossom Internat...",India,"[(llp, 3948), (pharmaceutical, 70), (chiplun, 3)]","[(karnataka, 7005), (bangalore, 5858), (in, 51..."
8,S1-497090852,75896,UA Technotronics Pvt Ltd,"Door No Tg Block 29, Jains Alpine Meadows Temp...",India,"[(ltd, 14974), (pvt, 12246), (ua, 14), (techno...","[(no, 36025), (tamil, 6401), (nadu, 6399), (bl..."
9,S1-189147360,74604,Ambassador & Co,"Plot No. 63 To 70, Tg/F, Mohit Nagar Kakrola, ...",India,"[(co, 2755), (ambassador, 2)]","[(no, 36025), (nagar, 15610), (west, 13357), (..."


In [44]:
for cutoff in [1000, 5000, 10000, 50000, 100000]:
    total_from_heavy = (
        threshold_2_counts[
            threshold_2_counts > cutoff
        ].sum()
    )

    print(
        f"> {cutoff:6,d} candidates: "
        f"{(threshold_2_counts > cutoff).sum():6,d} S1s | "
        f"{total_from_heavy:,} candidate assignments"
    )

>  1,000 candidates:  2,581 S1s | 8,938,916 candidate assignments
>  5,000 candidates:    209 S1s | 5,477,489 candidate assignments
> 10,000 candidates:    120 S1s | 4,869,339 candidate assignments
> 50,000 candidates:     41 S1s | 3,369,569 candidate assignments
> 100,000 candidates:      4 S1s | 1,051,415 candidate assignments


In [45]:
heavy_indices = set(
    heavy_s1.head(10)["s1_index"].astype(int)
)

heavy_records = validation_s1.iloc[
    list(heavy_indices)
]

heavy_name_tokens = set()
heavy_address_tokens = set()

for row in heavy_records.itertuples(index=False):
    heavy_name_tokens.update(
        get_tokens(row.s1_name)
    )
    heavy_address_tokens.update(
        get_tokens(row.s1_address)
    )

print(
    "Heavy-S1 name tokens:",
    len(heavy_name_tokens)
)

print(
    "Heavy-S1 address tokens:",
    len(heavy_address_tokens)
)

Heavy-S1 name tokens: 26
Heavy-S1 address tokens: 103


In [46]:
target_name_frequency = Counter()
target_address_frequency = Counter()

def scan_target_token_frequency(
    source_path,
    source_name
):
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            name_tokens = (
                get_tokens(row.business_name)
                &
                heavy_name_tokens
            )

            address_tokens = (
                get_tokens(row.business_address)
                &
                heavy_address_tokens
            )

            target_name_frequency.update(
                name_tokens
            )

            target_address_frequency.update(
                address_tokens
            )

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

In [47]:
scan_target_token_frequency(
    SOURCE2,
    "S2"
)

scan_target_token_frequency(
    SOURCE3,
    "S3"
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [48]:
heavy_token_comparison = []

for token in sorted(
    heavy_name_tokens
):
    heavy_token_comparison.append({
        "field": "name",
        "token": token,
        "validation_s1_frequency": (
            name_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_name_frequency.get(
                token,
                0
            )
        )
    })

for token in sorted(
    heavy_address_tokens
):
    heavy_token_comparison.append({
        "field": "address",
        "token": token,
        "validation_s1_frequency": (
            address_token_frequency.get(
                token,
                0
            )
        ),
        "s2_s3_target_frequency": (
            target_address_frequency.get(
                token,
                0
            )
        )
    })

heavy_token_comparison_df = pd.DataFrame(
    heavy_token_comparison
)

heavy_token_comparison_df = (
    heavy_token_comparison_df
    .sort_values(
        "s2_s3_target_frequency",
        ascending=False
    )
)

heavy_token_comparison_df.head(100)

,field,token,validation_s1_frequency,s2_s3_target_frequency
97,address,no,36025,1827811
105,address,road,45852,1288416
10,name,limited,52308,1202275
16,name,private,43192,1167992
12,name,ltd,14974,826245
...,...,...,...,...
15,name,pharmaceutical,70,2639
62,address,col,58,2049
44,address,aa,42,1881
87,address,mandvi,23,1076


In [49]:
heavy_s1[
    [
        "s1_id",
        "candidate_count",
        "s1_name",
        "s1_address",
        "name_token_frequencies",
        "address_token_frequencies"
    ]
].head(10)

,s1_id,candidate_count,s1_name,s1_address,name_token_frequencies,address_token_frequencies
0,S1-781739349,263003,Vizag Opticals Private Limited,"Shop 3Ab Jariwala Building, Mumbai City, Mumba...","[(limited, 52308), (private, 43192), (vizag, 8...","[(road, 45852), (maharashtra, 19232), (city, 1..."
1,S1-816511031,262924,Innovative Industries Pvt Ltd,27/28 Floor 6Th Plot 132/134 Aishabai Building...,"[(ltd, 14974), (pvt, 12246), (industries, 1913...","[(street, 31080), (maharashtra, 19232), (floor..."
2,S1-368134184,262764,Aes Tradecom Pvt Ltd,"176/1, Second Floor, Hari Nagar Ashram, Null, ...","[(ltd, 14974), (pvt, 12246), (tradecom, 33), (...","[(floor, 16306), (nagar, 15610), (delhi, 12180..."
3,S1-37009159,262724,Lucknow Estates Pvt Ltd,"Lig-49, Aishbaag Pill Col, Old Labour Colony N...","[(ltd, 14974), (pvt, 12246), (estates, 127), (...","[(pradesh, 11329), (uttar, 7370), (colony, 508..."
4,S1-652967049,97247,Future Silver Tech,"Sri Skanda No. 123, 1St Floor, 1St Cross, Aa C...","[(tech, 1372), (silver, 1222), (future, 449)]","[(road, 45852), (no, 36025), (floor, 16306), (..."
5,S1-353569058,95515,Astron Enn Pvt Ltd,"Survey No.321/Ak, Village Miroli, Taluka: Dask...","[(ltd, 14974), (pvt, 12246), (astron, 5), (enn...","[(no, 36025), (gujarat, 5568), (in, 5143), (vi..."
6,S1-301584609,95232,Ss Software Private Limited,"Gj-02 Sdf-7 Seepz, Maharashtra, Mumbai, Andher...","[(limited, 52308), (private, 43192), (software...","[(maharashtra, 19232), (mumbai, 8730), (andher..."
7,S1-492263052,93943,Chiplun Pharmaceutical LLP,"#330, Whitecity Layout Opp To Blossom Internat...","[(llp, 3948), (pharmaceutical, 70), (chiplun, 3)]","[(karnataka, 7005), (bangalore, 5858), (in, 51..."
8,S1-497090852,75896,UA Technotronics Pvt Ltd,"Door No Tg Block 29, Jains Alpine Meadows Temp...","[(ltd, 14974), (pvt, 12246), (ua, 14), (techno...","[(no, 36025), (tamil, 6401), (nadu, 6399), (bl..."
9,S1-189147360,74604,Ambassador & Co,"Plot No. 63 To 70, Tg/F, Mohit Nagar Kakrola, ...","[(co, 2755), (ambassador, 2)]","[(no, 36025), (nagar, 15610), (west, 13357), (..."


In [50]:
heavy_pair_keys = set()

for row in heavy_records.itertuples(index=False):

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    if len(name_tokens) >= 2:
        heavy_pair_keys.update(
            combinations(name_tokens, 2)
        )

    if len(address_tokens) >= 2:
        heavy_pair_keys.update(
            combinations(address_tokens, 2)
        )

print("Heavy S1 token pairs:", len(heavy_pair_keys))

Heavy S1 token pairs: 858


In [51]:
target_pair_frequency = Counter()

def scan_target_pair_frequency(source_path, source_name):

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            name_tokens = get_tokens(
                row.business_name
            )

            address_tokens = get_tokens(
                row.business_address
            )

            name_pairs = combinations(
                sorted(name_tokens),
                2
            )

            address_pairs = combinations(
                sorted(address_tokens),
                2
            )

            for pair in name_pairs:

                if pair in heavy_pair_keys:
                    target_pair_frequency[pair] += 1

            for pair in address_pairs:

                if pair in heavy_pair_keys:
                    target_pair_frequency[pair] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{chunk_number * CHUNK_SIZE:,} rows processed"
            )

In [52]:
scan_target_pair_frequency(
    SOURCE2,
    "S2"
)

scan_target_pair_frequency(
    SOURCE3,
    "S3"
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [54]:
# Build STRING-based pair frequencies specifically for the
# heavy S1 records so the keys match heavy_pair_keys.

heavy_name_pair_frequency = Counter()
heavy_address_pair_frequency = Counter()

for row in validation_s1.itertuples(index=False):

    name_tokens = sorted(
        get_tokens(row.s1_name)
    )

    address_tokens = sorted(
        get_tokens(row.s1_address)
    )

    if len(name_tokens) >= 2:
        heavy_name_pair_frequency.update(
            combinations(name_tokens, 2)
        )

    if len(address_tokens) >= 2:
        heavy_address_pair_frequency.update(
            combinations(address_tokens, 2)
        )


heavy_pair_comparison = []

for pair in heavy_pair_keys:

    name_freq = heavy_name_pair_frequency.get(
        pair,
        0
    )

    address_freq = heavy_address_pair_frequency.get(
        pair,
        0
    )

    # Keep the two fields separate.
    heavy_pair_comparison.append({
        "token_1": pair[0],
        "token_2": pair[1],
        "name_s1_frequency": name_freq,
        "address_s1_frequency": address_freq,
        "target_frequency": (
            target_pair_frequency.get(pair, 0)
        )
    })


heavy_pair_comparison_df = (
    pd.DataFrame(heavy_pair_comparison)
    .sort_values(
        "target_frequency",
        ascending=False
    )
)

heavy_pair_comparison_df.head(100)

,token_1,token_2,name_s1_frequency,address_s1_frequency,target_frequency
423,limited,private,43134,52,716685
113,no,road,3,8575,357042
697,delhi,new,535,7655,318957
823,ltd,pvt,12246,92,312306
376,floor,no,0,7760,305460
...,...,...,...,...,...
388,colony,pradesh,0,964,22322
474,city,plot,0,836,22134
16,ahmedabad,no,0,366,21961
25,bengaluru,no,0,244,21784


In [55]:
heavy_pair_comparison_df[
    [
        "token_1",
        "token_2",
        "name_s1_frequency",
        "address_s1_frequency",
        "target_frequency"
    ]
].head(50)

,token_1,token_2,name_s1_frequency,address_s1_frequency,target_frequency
423,limited,private,43134,52,716685
113,no,road,3,8575,357042
697,delhi,new,535,7655,318957
823,ltd,pvt,12246,92,312306
376,floor,no,0,7760,305460
457,nagar,no,3,6923,268874
82,no,plot,0,7642,263255
475,door,no,0,814,220843
142,delhi,no,0,4547,219713
27,floor,road,2,4570,155666


In [56]:
PAIR_S1_THRESHOLD = 2

active_name_pairs = {
    pair
    for pair, freq in name_pair_frequency.items()
    if 0 < freq <= PAIR_S1_THRESHOLD
}

active_address_pairs = {
    pair
    for pair, freq in address_pair_frequency.items()
    if 0 < freq <= PAIR_S1_THRESHOLD
}

print("Active name pairs:", len(active_name_pairs))
print("Active address pairs:", len(active_address_pairs))

Active name pairs: 340947
Active address pairs: 2183614


In [57]:
target_name_pair_frequency_2 = Counter()
target_address_pair_frequency_2 = Counter()


def scan_active_pair_frequency(
    source_path,
    source_name
):
    total_rows = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            if len(name_token_ids) >= 2:

                for pair in combinations(
                    sorted(name_token_ids),
                    2
                ):
                    if pair in active_name_pairs:
                        target_name_pair_frequency_2[pair] += 1

            if len(address_token_ids) >= 2:

                for pair in combinations(
                    sorted(address_token_ids),
                    2
                ):
                    if pair in active_address_pairs:
                        target_address_pair_frequency_2[pair] += 1

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

In [58]:
scan_active_pair_frequency(
    SOURCE2,
    "S2"
)

scan_active_pair_frequency(
    SOURCE3,
    "S3"
)

S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


In [60]:
target_pair_frequency_2 = Counter()

target_pair_frequency_2.update(
    target_name_pair_frequency_2
)

target_pair_frequency_2.update(
    target_address_pair_frequency_2
)

print(
    "Active pairs with target occurrences:",
    len(target_pair_frequency_2)
)

print(
    target_pair_frequency_2.most_common(30)
)

Active pairs with target occurrences: 1033249
[((460, 9535), 92153), ((657, 11627), 48338), ((770, 9455), 45594), ((714, 4485), 37818), ((304, 2249), 29352), ((5053, 8797), 28392), ((338, 3516), 22703), ((3364, 6454), 22206), ((617, 9095), 21211), ((1912, 11627), 20044), ((3364, 3514), 19291), ((2593, 6457), 18930), ((2, 24001), 18427), ((1357, 9535), 18323), ((2, 20244), 18154), ((4, 24001), 18133), ((4, 20244), 18075), ((417, 9535), 17979), ((29, 30430), 16396), ((51, 30430), 16183), ((813, 5530), 15969), ((1637, 10439), 15442), ((2777, 57526), 15430), ((770, 16451), 15219), ((4527, 6881), 15062), ((1577, 11743), 14154), ((2593, 6454), 13381), ((2256, 10439), 13287), ((3454, 14555), 12630), ((1125, 14555), 12626)]


In [61]:
TARGET_PAIR_THRESHOLDS = [
    10,
    50,
    100,
    500,
    1000,
    5000
]

In [65]:
# ---------------------------------------------------------
# Rebuild the true-target lookup
# ---------------------------------------------------------

true_target_to_s1_idx = {}

for s1_id, target_ids in validation_ground_truth.items():
    s1_idx = s1_id_to_index[s1_id]

    for target_id in target_ids:
        true_target_to_s1_idx[target_id] = s1_idx

print(
    "Unique validation true targets:",
    len(true_target_to_s1_idx)
)

Unique validation true targets: 764045


In [67]:
true_target_min_pair_freq = {
    target_id: np.inf
    for target_id in true_target_to_s1_idx
}

print(
    "Targets to evaluate:",
    len(true_target_min_pair_freq)
)

Targets to evaluate: 764045


In [68]:
def evaluate_true_target_pair_selectivity(
    source_path,
    source_name
):
    rows_scanned = 0
    true_targets_seen = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        rows_in_chunk = len(chunk)
        rows_scanned += rows_in_chunk

        for row in chunk.itertuples(index=False):

            target_id = row.entity_id

            if target_id not in true_target_min_pair_freq:
                continue

            true_targets_seen += 1

            min_freq = true_target_min_pair_freq[
                target_id
            ]

            # -----------------------------
            # NAME PAIRS
            # -----------------------------
            name_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_name
                )
                if token in token_to_id
            ]

            if len(name_token_ids) >= 2:

                for pair in combinations(
                    sorted(name_token_ids),
                    2
                ):

                    if pair not in active_name_pairs:
                        continue

                    freq = target_name_pair_frequency_2.get(
                        pair,
                        np.inf
                    )

                    if freq < min_freq:
                        min_freq = freq

            # -----------------------------
            # ADDRESS PAIRS
            # -----------------------------
            address_token_ids = [
                token_to_id[token]
                for token in get_tokens(
                    row.business_address
                )
                if token in token_to_id
            ]

            if len(address_token_ids) >= 2:

                for pair in combinations(
                    sorted(address_token_ids),
                    2
                ):

                    if pair not in active_address_pairs:
                        continue

                    freq = target_address_pair_frequency_2.get(
                        pair,
                        np.inf
                    )

                    if freq < min_freq:
                        min_freq = freq

            true_target_min_pair_freq[
                target_id
            ] = min_freq

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{rows_scanned:,} rows scanned | "
                f"{true_targets_seen:,} true targets seen"
            )

    print(
        f"{source_name} COMPLETE: "
        f"{rows_scanned:,} rows scanned | "
        f"{true_targets_seen:,} true targets seen"
    )

In [69]:
evaluate_true_target_pair_selectivity(
    SOURCE2,
    "S2"
)

evaluate_true_target_pair_selectivity(
    SOURCE3,
    "S3"
)

S2: 1,000,000 rows scanned | 73,785 true targets seen
S2: 2,000,000 rows scanned | 147,069 true targets seen
S2: 3,000,000 rows scanned | 220,347 true targets seen
S2: 4,000,000 rows scanned | 293,944 true targets seen
S2: 5,000,000 rows scanned | 367,347 true targets seen
S2 COMPLETE: 5,034,616 rows scanned | 369,899 true targets seen
S3: 1,000,000 rows scanned | 75,131 true targets seen
S3: 2,000,000 rows scanned | 149,580 true targets seen
S3: 3,000,000 rows scanned | 224,270 true targets seen
S3: 4,000,000 rows scanned | 298,812 true targets seen
S3: 5,000,000 rows scanned | 373,310 true targets seen
S3 COMPLETE: 5,285,603 rows scanned | 394,146 true targets seen


In [70]:
target_selectivity_results = []

for threshold in TARGET_PAIR_THRESHOLDS:

    surviving = sum(
        freq <= threshold
        for freq in true_target_min_pair_freq.values()
    )

    target_selectivity_results.append({
        "target_pair_frequency_threshold": threshold,
        "true_matches_surviving": surviving,
        "total_true_matches": total_true_matches,
        "candidate_recall": (
            surviving / total_true_matches
        )
    })

target_selectivity_df = pd.DataFrame(
    target_selectivity_results
)

target_selectivity_df

,target_pair_frequency_threshold,true_matches_surviving,total_true_matches,candidate_recall
0,10,477417,764045,0.624855
1,50,588614,764045,0.770392
2,100,603261,764045,0.789562
3,500,613453,764045,0.802902
4,1000,616559,764045,0.806967
5,5000,623096,764045,0.815523


In [71]:
import numpy as np
import pandas as pd

# Load / rebuild current blocker true targets if needed
# This should represent the targets captured by:
# exact name + name token <=10 + address token <=10 + pair <=2

current_captured_true_targets = set()

# If you already have this set saved, load/use it here.
# Example:
# current_captured_true_targets = captured_true_target_ids

thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_targets = len(true_target_to_s1_idx)

rows = []

for threshold in thresholds:
    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    newly_recovered = target_route_targets - current_captured_true_targets
    union_targets = current_captured_true_targets | target_route_targets

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "new_true_matches_added": len(newly_recovered),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_targets
    })

target_union_df = pd.DataFrame(rows)
target_union_df

,target_pair_frequency_threshold,target_route_true_matches,new_true_matches_added,union_true_matches,union_recall
0,10,477417,477417,477417,0.624855
1,50,588614,588614,588614,0.770392
2,100,603261,603261,603261,0.789562
3,500,613453,613453,613453,0.802902
4,1000,616559,616559,616559,0.806967
5,5000,623096,623096,623096,0.815523


In [73]:
import pickle
import os

path = "/Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/blocking_threshold_10_captured.pkl"

with open(path, "rb") as f:
    obj = pickle.load(f)

print("Object type:", type(obj))

if isinstance(obj, dict):
    print("Dictionary keys:")
    for k, v in obj.items():
        print(f"  {k}: type={type(v)}, size={len(v) if hasattr(v, '__len__') else 'NA'}")
else:
    print("Size:", len(obj) if hasattr(obj, "__len__") else "NA")

Object type: <class 'set'>
Size: 572131


In [74]:
def find_target_id_set(obj):
    """
    Find a set/list containing S2-/S3- target IDs.
    """
    if isinstance(obj, (set, list, tuple)):
        sample = list(obj)[:20]
        if any(
            isinstance(x, str) and (x.startswith("S2-") or x.startswith("S3-"))
            for x in sample
        ):
            return set(obj)

    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (set, list, tuple)):
                sample = list(value)[:20]

                if any(
                    isinstance(x, str)
                    and (x.startswith("S2-") or x.startswith("S3-"))
                    for x in sample
                ):
                    print("Using key:", key)
                    return set(value)

    raise ValueError("Could not find S2/S3 target ID set in pickle.")


current_captured_true_targets = find_target_id_set(obj)

print(
    "Current blocker captured true targets:",
    len(current_captured_true_targets)
)

Current blocker captured true targets: 572131


In [75]:
thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_targets = len(true_target_to_s1_idx)

rows = []

for threshold in thresholds:

    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    newly_recovered = (
        target_route_targets
        - current_captured_true_targets
    )

    union_targets = (
        current_captured_true_targets
        | target_route_targets
    )

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "current_blocker_true_matches": len(current_captured_true_targets),
        "new_true_matches_added": len(newly_recovered),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_targets
    })

target_union_df = pd.DataFrame(rows)

target_union_df

,target_pair_frequency_threshold,target_route_true_matches,current_blocker_true_matches,new_true_matches_added,union_true_matches,union_recall
0,10,477417,572131,66952,639083,0.836447
1,50,588614,572131,106476,678607,0.888177
2,100,603261,572131,113329,685460,0.897146
3,500,613453,572131,118655,690786,0.904117
4,1000,616559,572131,120210,692341,0.906152
5,5000,623096,572131,123152,695283,0.910003


In [77]:
from pathlib import Path
import pickle

filename = "blocking_experiment_02_pair_state.pkl"

candidates = [
    Path("output") / filename,
    Path("../output") / filename,
    Path("../../output") / filename,
]

pair_state_path = next((p.resolve() for p in candidates if p.exists()), None)

if pair_state_path is None:
    raise FileNotFoundError(
        f"{filename} not found. Checked:\n" +
        "\n".join(str(p.resolve()) for p in candidates)
    )

print("Found:", pair_state_path)

with open(pair_state_path, "rb") as f:
    pair_state = pickle.load(f)

print("Type:", type(pair_state))

if isinstance(pair_state, dict):
    for k, v in pair_state.items():
        size = len(v) if hasattr(v, "__len__") else "NA"
        print(f"{k}: {type(v)}, size={size}")

Found: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/output/blocking_experiment_02_pair_state.pkl
Type: <class 'dict'>
pair_true_hits: <class 'dict'>, size=6
total_true_matches: <class 'int'>, size=NA


In [78]:
pair_true_hits = pair_state["pair_true_hits"]

print("Threshold keys:", pair_true_hits.keys())

for threshold, value in pair_true_hits.items():
    print(
        f"\nThreshold <= {threshold}"
        f"\nType: {type(value)}"
        f"\nSize: {len(value) if hasattr(value, '__len__') else 'NA'}"
    )

    if isinstance(value, (set, list, tuple)):
        print("First 5:", list(value)[:5])
    elif isinstance(value, dict):
        print("First 5 keys:", list(value.keys())[:5])

Threshold keys: dict_keys([2, 5, 10, 20, 50, 100])

Threshold <= 2
Type: <class 'set'>
Size: 627313
First 5: ['S2-464926415', 'S3-234542742', 'S3-155508462', 'S3-995633822', 'S3-806104210']

Threshold <= 5
Type: <class 'set'>
Size: 658646
First 5: ['S2-464926415', 'S3-155508462', 'S3-486946035', 'S2-393836139', 'S2-490645791']

Threshold <= 10
Type: <class 'set'>
Size: 676178
First 5: ['S2-464926415', 'S3-155508462', 'S3-486946035', 'S2-393836139', 'S2-490645791']

Threshold <= 20
Type: <class 'set'>
Size: 688143
First 5: ['S2-464926415', 'S3-155508462', 'S3-486946035', 'S2-393836139', 'S2-490645791']

Threshold <= 50
Type: <class 'set'>
Size: 697760
First 5: ['S2-464926415', 'S3-155508462', 'S3-486946035', 'S2-393836139', 'S2-490645791']

Threshold <= 100
Type: <class 'set'>
Size: 704326
First 5: ['S2-464926415', 'S3-155508462', 'S3-486946035', 'S2-393836139', 'S2-490645791']


In [79]:
# True targets captured by the current blocker
# (exact name + name-token <=10 + address-token <=10)
print("Current blocker:", len(current_captured_true_targets))

# True targets captured by pair blocker <=2
pair2_true_targets = pair_true_hits[2]
print("Pair <=2:", len(pair2_true_targets))

# Union = actual current blocker + pair<=2 baseline
pair2_union_true_targets = (
    current_captured_true_targets
    | pair2_true_targets
)

print("Union:", len(pair2_union_true_targets))

Current blocker: 572131
Pair <=2: 627313
Union: 696931


In [80]:
overlap = (
    current_captured_true_targets
    & pair2_true_targets
)

print("Overlap:", len(overlap))
print("Expected overlap: 502513")

print(
    "Recall:",
    len(pair2_union_true_targets) / pair_state["total_true_matches"]
)

Overlap: 502513
Expected overlap: 502513
Recall: 0.91215962410591


In [81]:
thresholds = [10, 50, 100, 500, 1000, 5000]

total_true_matches = pair_state["total_true_matches"]

rows = []

for threshold in thresholds:

    target_route_targets = {
        target_id
        for target_id, freq in true_target_min_pair_freq.items()
        if freq <= threshold
    }

    new_matches = (
        target_route_targets
        - pair2_union_true_targets
    )

    union_targets = (
        pair2_union_true_targets
        | target_route_targets
    )

    rows.append({
        "target_pair_frequency_threshold": threshold,
        "target_route_true_matches": len(target_route_targets),
        "new_true_matches_added": len(new_matches),
        "union_true_matches": len(union_targets),
        "union_recall": len(union_targets) / total_true_matches
    })

target_union_df = pd.DataFrame(rows)

target_union_df

,target_pair_frequency_threshold,target_route_true_matches,new_true_matches_added,union_true_matches,union_recall
0,10,477417,3964,700895,0.917348
1,50,588614,8543,705474,0.923341
2,100,603261,9952,706883,0.925185
3,500,613453,12491,709422,0.928508
4,1000,616559,13521,710452,0.929856
5,5000,623096,15517,712448,0.932469


In [95]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np

TARGET_PAIR_THRESHOLDS = [10, 50, 100, 500, 1000, 5000]
MAX_TARGET_THRESHOLD = max(TARGET_PAIR_THRESHOLDS)

# Robust source paths
def resolve_source_path(filename):
    candidates = [
        Path("data/train") / filename,
        Path("../data/train") / filename,
        Path("../../data/train") / filename,
    ]

    for p in candidates:
        if p.exists():
            return p.resolve()

    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n" +
        "\n".join(str(p.resolve()) for p in candidates)
    )


SOURCE2_PATH = resolve_source_path("train_source2.tsv")
SOURCE3_PATH = resolve_source_path("train_source3.tsv")

print("S2:", SOURCE2_PATH)
print("S3:", SOURCE3_PATH)


def evaluate_target_pair_candidate_volume(source_path, source_name):

    totals = {t: 0 for t in TARGET_PAIR_THRESHOLDS}
    row_counts_sample = {t: [] for t in TARGET_PAIR_THRESHOLDS}

    total_rows = 0
    zero_candidate_rows = 0
    max_candidates = {t: 0 for t in TARGET_PAIR_THRESHOLDS}

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            total_rows += 1

            # For this target row:
            # S1 index -> minimum target-side pair frequency
            min_freq_by_s1 = {}

            # --------------------------------------------------
            # NAME PAIRS
            # --------------------------------------------------
            row_name_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_name)
                if token in token_to_id
            ]

            if len(row_name_ids) >= 2:

                for pair in combinations(
                    sorted(row_name_ids), 2
                ):

                    freq = target_name_pair_frequency_2.get(pair)

                    if freq is None or freq > MAX_TARGET_THRESHOLD:
                        continue

                    s1_postings = name_pair_to_s1.get(pair, [])

                    for s1_idx in s1_postings:

                        previous = min_freq_by_s1.get(s1_idx)

                        if previous is None or freq < previous:
                            min_freq_by_s1[s1_idx] = freq

            # --------------------------------------------------
            # ADDRESS PAIRS
            # --------------------------------------------------
            row_address_ids = [
                token_to_id[token]
                for token in get_tokens(row.business_address)
                if token in token_to_id
            ]

            if len(row_address_ids) >= 2:

                for pair in combinations(
                    sorted(row_address_ids), 2
                ):

                    freq = target_address_pair_frequency_2.get(pair)

                    if freq is None or freq > MAX_TARGET_THRESHOLD:
                        continue

                    s1_postings = address_pair_to_s1.get(pair, [])

                    for s1_idx in s1_postings:

                        previous = min_freq_by_s1.get(s1_idx)

                        if previous is None or freq < previous:
                            min_freq_by_s1[s1_idx] = freq

            # --------------------------------------------------
            # Count unique (target, S1) candidates by threshold
            # --------------------------------------------------
            counts = [0] * len(TARGET_PAIR_THRESHOLDS)

            for freq in min_freq_by_s1.values():

                if freq <= 10:
                    counts[0] += 1
                elif freq <= 50:
                    counts[1] += 1
                elif freq <= 100:
                    counts[2] += 1
                elif freq <= 500:
                    counts[3] += 1
                elif freq <= 1000:
                    counts[4] += 1
                elif freq <= 5000:
                    counts[5] += 1

            # Convert bucket counts into cumulative threshold counts
            for i in range(1, len(counts)):
                counts[i] += counts[i - 1]

            if len(min_freq_by_s1) == 0:
                zero_candidate_rows += 1

            for i, threshold in enumerate(TARGET_PAIR_THRESHOLDS):

                c = counts[i]

                totals[threshold] += c

                if c > max_candidates[threshold]:
                    max_candidates[threshold] = c

                # Deterministic sample: every 1000th target row
                if total_rows % 1000 == 0:
                    row_counts_sample[threshold].append(c)

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: "
                f"{total_rows:,} rows processed"
            )

    results = []

    for threshold in TARGET_PAIR_THRESHOLDS:

        sample = np.array(
            row_counts_sample[threshold],
            dtype=np.int64
        )

        results.append({
            "source": source_name,
            "target_pair_frequency_threshold": threshold,
            "candidate_pairs": totals[threshold],
            "avg_candidates_per_target": (
                totals[threshold] / total_rows
                if total_rows else 0
            ),
            "median_candidates_per_target_sample": (
                np.median(sample) if len(sample) else 0
            ),
            "p95_candidates_per_target_sample": (
                np.percentile(sample, 95) if len(sample) else 0
            ),
            "max_candidates_per_target": max_candidates[threshold],
            "zero_candidate_target_pct": (
                100 * zero_candidate_rows / total_rows
                if total_rows else 0
            )
        })

    return pd.DataFrame(results)


# --------------------------------------------------------------
# Run S2 and S3
# --------------------------------------------------------------

target_candidate_s2_df = evaluate_target_pair_candidate_volume(
    SOURCE2_PATH,
    "S2"
)

target_candidate_s3_df = evaluate_target_pair_candidate_volume(
    SOURCE3_PATH,
    "S3"
)

target_candidate_volume_df = pd.concat(
    [
        target_candidate_s2_df,
        target_candidate_s3_df
    ],
    ignore_index=True
)

target_candidate_volume_df

S2: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/data/train/train_source2.tsv
S3: /Users/bhavikesh/Documents/Hackathon/Amazon_ml/amazon-ml-challenge/data/train/train_source3.tsv
S2: 1,000,000 rows processed
S2: 2,000,000 rows processed
S2: 3,000,000 rows processed
S2: 4,000,000 rows processed
S2: 5,000,000 rows processed
S3: 1,000,000 rows processed
S3: 2,000,000 rows processed
S3: 3,000,000 rows processed
S3: 4,000,000 rows processed
S3: 5,000,000 rows processed


,source,target_pair_frequency_threshold,candidate_pairs,avg_candidates_per_target,median_candidates_per_target_sample,p95_candidates_per_target_sample,max_candidates_per_target,zero_candidate_target_pct
0,S2,10,765722,0.152091,0.0,1.0,7,40.971188
1,S2,50,4025388,0.799542,0.0,3.0,26,40.971188
2,S2,100,5937608,1.179357,1.0,5.0,42,40.971188
3,S2,500,7196551,1.429414,1.0,5.0,43,40.971188
4,S2,1000,7369288,1.463724,1.0,5.0,43,40.971188
5,S2,5000,7590191,1.507601,1.0,6.0,43,40.971188
6,S3,10,694145,0.131327,0.0,1.0,7,43.388559
7,S3,50,3467869,0.656097,0.0,3.0,23,43.388559
8,S3,100,5146548,0.973692,0.0,4.0,43,43.388559
9,S3,500,6462128,1.222590,1.0,5.0,47,43.388559


In [96]:
target_candidate_volume_summary = (
    target_candidate_volume_df
    .groupby("target_pair_frequency_threshold", as_index=False)
    .agg({
        "candidate_pairs": "sum",
        "avg_candidates_per_target": "mean",
        "median_candidates_per_target_sample": "mean",
        "p95_candidates_per_target_sample": "mean",
        "max_candidates_per_target": "max"
    })
)

target_candidate_volume_summary

,target_pair_frequency_threshold,candidate_pairs,avg_candidates_per_target,median_candidates_per_target_sample,p95_candidates_per_target_sample,max_candidates_per_target
0,10,1459867,0.141709,0.0,1.0,7
1,50,7493257,0.727820,0.0,3.0,26
2,100,11084156,1.076524,0.5,4.5,43
3,500,13658679,1.326002,1.0,5.0,47
4,1000,14135515,1.371924,1.0,5.0,47
5,5000,15067670,1.461144,1.0,5.5,47


In [97]:
# Target-side true targets at threshold 500
target_500_true_targets = {
    target_id
    for target_id, freq in true_target_min_pair_freq.items()
    if freq <= 500
}

# Full blocker true-target coverage
blocker_500_true_targets = (
    pair2_union_true_targets
    | target_500_true_targets
)

total_true_matches = pair_state["total_true_matches"]

print("Baseline pair<=2 union:", len(pair2_union_true_targets))
print("Target route <=500:", len(target_500_true_targets))
print("Final blocker <=500:", len(blocker_500_true_targets))

print(
    "Recall:",
    len(blocker_500_true_targets) / total_true_matches
)

print(
    "Remaining missed:",
    total_true_matches - len(blocker_500_true_targets)
)


Baseline pair<=2 union: 696931
Target route <=500: 613453
Final blocker <=500: 709422
Recall: 0.9285081376096957
Remaining missed: 54623


In [99]:
from pathlib import Path

OUTPUT_DIR = Path(OUTPUT_DIR)

blocking_artifact_path = OUTPUT_DIR / "blocking_artifacts_v1.pkl"

with open(blocking_artifact_path, "wb") as f:
    pickle.dump(
        blocking_artifact,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved:", blocking_artifact_path)

Saved: ../output/blocking_artifacts_v1.pkl


In [ ]:
# Complete blocking artifact for downstream notebooks

blocking_artifact.update({

    # Token vocabulary
    "token_to_id": token_to_id,

    # Exact normalized name
    "name_exact_map": dict(name_exact_map),

    # Single-token posting lists
    "name_token_to_s1": dict(name_token_to_s1),
    "address_token_to_s1": dict(address_token_to_s1),

    # IMPORTANT:
    # These frequencies are required to reproduce the
    # <=10 single-token blocking rule.
    "name_token_frequency": dict(name_token_frequency),
    "address_token_frequency": dict(address_token_frequency),

    # Pair blocking
    "name_pair_to_s1": dict(name_pair_to_s1),
    "address_pair_to_s1": dict(address_pair_to_s1),

    # Active S1-side pairs
    "active_name_pairs": active_name_pairs,
    "active_address_pairs": active_address_pairs,

    # Target-side pair frequencies
    "target_name_pair_frequency_2": target_name_pair_frequency_2,
    "target_address_pair_frequency_2": target_address_pair_frequency_2,

    # Final blocking thresholds
    "NAME_TOKEN_FREQUENCY_THRESHOLD": 10,
    "ADDRESS_TOKEN_FREQUENCY_THRESHOLD": 10,
    "PAIR_S1_THRESHOLD": 2,
    "TARGET_PAIR_THRESHOLD": 500,
})

blocking_artifact_path = OUTPUT_DIR / "blocking_artifacts_v1.pkl"

with open(
    blocking_artifact_path,
    "wb"
) as f:
    pickle.dump(
        blocking_artifact,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("Saved complete blocking artifact:")
print(blocking_artifact_path)

print("\nKeys saved:")
for key in sorted(blocking_artifact.keys()):
    print(" -", key)

Saved complete blocking artifact:
../output/blocking_artifacts_v1.pkl

Keys saved:
 - ADDRESS_TOKEN_FREQUENCY_THRESHOLD
 - NAME_TOKEN_FREQUENCY_THRESHOLD
 - PAIR_S1_THRESHOLD
 - TARGET_PAIR_THRESHOLD
 - active_address_pairs
 - active_name_pairs
 - address_pair_to_s1
 - address_token_frequency
 - address_token_ids
 - address_token_to_s1
 - name_exact_map
 - name_pair_to_s1
 - name_token_frequency
 - name_token_ids
 - name_token_to_s1
 - target_address_pair_frequency_2
 - target_name_pair_frequency_2
 - token_to_id
 - validation_s1_ids
